# 10 - Relational SQL Database Ingestion for RAG

> Complete guide and in-depth Arabic documentation: [`README.md`](./README.md)

Bridges normalized 3NF relational databases and RAG vector search via cross-table JOIN serialization.

## 1. Database Connection via SQLDatabase

In [ ]:
import os
import sqlite3
from pathlib import Path
from typing import List, Dict, Any

from langchain_core.documents import Document
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

print("SQL Database utilities and Document loaders imported successfully!")

## 2. Table-by-Table Ingestion with SQLDatabaseLoader

In [ ]:
from pathlib import Path
data_dir = Path('data') if Path('data').exists() else Path('../../data')
db_file = (data_dir / 'databases' / 'company.db').resolve()

# التأكد من وجود قاعدة البيانات التي تم تغذيتها مركزياً
print(f"[+] SQLite Database located at: {db_file} (Exists: {db_file.exists()})")


## 3. Cross-Table Relational JOIN Serialization
Joins normalized tables (employees, departments, projects) into complete entity profiles.

In [ ]:
db_uri = f"sqlite:///{db_file.as_posix()}"
db = SQLDatabase.from_uri(db_uri)

print(f"[+] Usable Tables: {db.get_usable_table_names()}")
print("\n--- Employees Table Schema & Sample Records ---")
print(db.get_table_info(["employees"]))

print("--- Projects Table Schema & Sample Records ---")
print(db.get_table_info(["projects"]))

## 4. Text-to-SQL vs Semantic Database Ingestion Tradeoffs

In [ ]:
sql_query = "SELECT name, role, department, salary FROM employees WHERE salary >= 95000"

loader = SQLDatabaseLoader(
    query=sql_query,
    db=db
)
query_docs = loader.load()

print(f"[+] Extracted {len(query_docs)} records from SQL query:\n")
for i, doc in enumerate(query_docs, 1):
    print(f"--- Record #{i} ---")
    print(doc.page_content)
    print()

## 5. Querying the Serialized Document Store

In [ ]:
def sql_to_documents(db_path: str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    documents: List[Document] = []

    # 1. استخراج أسماء الجداول من sqlite_master
    cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cur.fetchall()]

    for table_name in tables:
        # 2. استخراج بيانات الأعمدة عبر PRAGMA table_info
        cur.execute(f"PRAGMA table_info({table_name});")
        col_info = cur.fetchall()
        columns = [c[1] for c in col_info]

        # 3. استخراج الصفوف
        cur.execute(f"SELECT * FROM {table_name};")
        rows = cur.fetchall()

        # صياغة محتوى وصفي للجدول
        content_lines = [
            f"Database Table: {table_name}",
            f"Defined Columns: {', '.join(columns)}",
            f"Total Rows: {len(rows)}",
            "Sample Data:"
        ]
        for row in rows[:5]:
            row_str = " | ".join([f"{col}: {val}" for col, val in zip(columns, row)])
            content_lines.append(f"  - {row_str}")

        metadata = {
            "source": Path(db_path).name,
            "table_name": table_name,
            "row_count": len(rows),
            "columns_count": len(columns)
        }
        documents.append(Document(page_content="\n".join(content_lines), metadata=metadata))

    conn.close()
    return documents

table_docs = sql_to_documents(str(db_file))
print(f"[+] Generated {len(table_docs)} table schema documents.\n")
for doc in table_docs:
    print(f"=== Table: {doc.metadata['table_name']} ===")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}\n")

## 6. Architecture Comparison Matrix

In [ ]:
def serialize_relational_data(db_path: str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    query = """
    SELECT 
        e.name AS lead_name,
        e.role AS lead_role,
        e.department,
        p.name AS project_name,
        p.status AS project_status,
        p.budget AS project_budget
    FROM employees e
    INNER JOIN projects p ON e.id = p.lead_id
    """
    cur.execute(query)
    rows = cur.fetchall()
    documents: List[Document] = []

    for row in rows:
        lead_name, role, dept, proj_name, status, budget = row

        # صياغة نص متكامل للعلاقة
        content = (
            f"Project Overview: {proj_name}\n"
            f"Current Status: {status}\n"
            f"Approved Budget: ${budget:,.2f}\n"
            f"Project Lead: {lead_name} (Position: {role})\n"
            f"Managing Department: {dept}"
        )

        metadata = {
            "source": Path(db_path).name,
            "relation_type": "employee_leads_project",
            "project_name": proj_name,
            "project_status": status,
            "budget": budget,
            "lead_name": lead_name,
            "department": dept
        }
        documents.append(Document(page_content=content, metadata=metadata))

    conn.close()
    return documents

relational_documents = serialize_relational_data(str(db_file))
print(f"[+] Generated {len(relational_documents)} relational joined documents:\n")
for doc in relational_documents:
    print(f"--- {doc.metadata['project_name']} [{doc.metadata['project_status']}] ---")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}\n")

## Summary
Cross-table JOIN serialization provides self-contained context for semantic retrieval.